# **项目文件**

## 说明

本项目使用逻辑回归算法，分析学生的学生日常学习数据以及学习的相关信息，预测学生学业是否会失败。

# 代码

## 训练模型
运行代码可在目录中生成`student_failure_predict_model.joblib`(模型文件)和`student_failure_predict_model.pkl`(最优阈值)

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score,confusion_matrix

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import log_loss, roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


df2 = pd.read_csv("AI_SocialMedia_Student_Health_Dataset_clean.csv")
print(df2.head())

print(df2.shape)
print(df2.dtypes)
print("------------")
print(df2.isna().sum().sort_values(ascending=False).head())
print("duplicates:", df2.duplicated().sum())
print("-------------")
print(df2.describe().T[["mean", "std", "min", "max"]])
print(df2["Academic_Failure_Risk"].value_counts(normalize=True))

# df2["is_male"] = df2["Gender"].apply(lambda x: 1 if x == "Male" else 0)
# df2["is_female"] = df2["Gender"].apply(lambda x: 1 if x == "Female" else 0)
# df2["is_non_binary"] = df2["Gender"].apply(lambda x: 1 if x == "Non-binary" else 0)

# df2["is_high_school"] = df2["Education_Level"].apply(lambda x: 1 if x == "High School" else 0)
# df2["is_college"] = df2["Education_Level"].apply(lambda x: 1 if x == "College" else 0)
# df2["is_university"] = df2["Education_Level"].apply(lambda x: 1 if x == "University" else 0)

# df2["is_low_burnout"] = df2["Burnout_Level"].apply(lambda x: 1 if x == "Low" else 0)
# df2["is_moderate_burnout"] = df2["Burnout_Level"].apply(lambda x: 1 if x == "Moderate" else 0)
# df2["is_high_burnout"] = df2["Burnout_Level"].apply(lambda x: 1 if x == "High" else 0)
# df2["is_severe_burnout"] = df2["Burnout_Level"].apply(lambda x: 1 if x == "Severe" else 0)

# not_feature_cols = ["Academic_Failure_Risk", "Student_ID", "Gender","Education_Level", "Burnout_Level"]
# X = df2.drop(columns=not_feature_cols)

drop_raw = ["Academic_Failure_Risk","Student_ID"]
X = df2.drop(columns=drop_raw)
y = df2["Academic_Failure_Risk"]

categorical_cols = ["Gender","Education_Level", "Burnout_Level"]
num_cols = [c for c in X.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("categorical", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), categorical_cols)
    ]
)

print(X.dtypes)

y = df2["Academic_Failure_Risk"]

XTrain, X_temp, yTrain, y_temp = train_test_split(
    X, y, test_size=0.4,
    random_state=10,
    stratify=y
)

XValue, XTest, yValue, yTest = train_test_split(
    X_temp, y_temp, test_size=0.5,
    random_state=10,
    stratify=y_temp
)

model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)
model.fit(XTrain, yTrain)

pVal = model.predict_proba(XValue)[:, 1]
predVal05 = model.predict(XValue)




print("logloss", log_loss(yValue, pVal))
print("AUC", roc_auc_score(yValue, pVal))
print("BA", balanced_accuracy_score(yValue, predVal05))
print(confusion_matrix(yValue, predVal05))

MAX_FP = 0.15
thresholds = np.linspace(0.1, 0.9, 2000)

best_threshold = None
best_fp = 1.0

for t in thresholds:
    pred_val = (pVal >= t).astype(int)
    cm = confusion_matrix(yValue, pred_val)
    tn, fp, fn, tp = cm.ravel()

    fp_rate = fp / (tn + fp) if (tn + fp) > 0 else 1.0
    recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if fp_rate <= MAX_FP and recall_pos >=0.999:
        if fp_rate < best_fp:
            best_fp = fp_rate
            best_threshold = t

if best_threshold is None:
    best_threshold = 0.5

pred_val_best = (pVal >= best_threshold).astype(int)
cm_best = confusion_matrix(yValue, pred_val_best)
tn_b, fp_b, fn_b, tp_b = cm_best.ravel()
fp_rate_best = fp_b/(tn_b+fp_b) if (tn_b+fp_b)>0 else 0.0

print("ValBest")
print(f"MAX_FP={MAX_FP:.2f}")
print(f"best_threshold = {best_threshold:.4f}")
print(f"REAL_FP={fp_rate_best:.4f}")
print(cm_best)

print("Test")
pTest = model.predict_proba(XTest)[:, 1]
pred_test_final = (pTest >= best_threshold).astype(int)
cm_test = confusion_matrix(yTest, pred_test_final)
tn_t, fp_t, fn_t, tp_t = cm_test.ravel()

fp_rate_test = fp_t/(tn_t+fp_t) if (tn_t+fp_t)>0 else 0.0
recall_test = tp_t/(tp_t+fn_t) if (tp_t+fn_t)>0 else 0.0

print(cm_test)
print(f"TestFP:{fp_rate_test:.4f}")
print(f"recall_score:{recall_test:.4f}")
joblib.dump(model, "student_failure_predict_model.joblib")
joblib.dump(best_threshold, "logreg_best_threshold.pkl")

print("模型已保存:student_failure_predict_model.joblib")
print("最优阈值已保存:logreg_best_threshold.pkl")

## 使用模型
##### 保证已运行过上一个代码块，运行代码并填写学生信息，模型会输出学业是否成功的预测结果
**(模型预测仅供参考)**

In [ ]:
import pandas as pd
import joblib

def get_valid_input(prompt, valid_options):
    while True:
        user_input = input(f"[选择]{prompt}").strip()
        if user_input in valid_options:
            return user_input
        print(f"无效输入，请输入 {valid_options} 中的一个选项")

def get_valid_number(prompt, convert_func=float):
    try:
        if convert_func == int:
            return int(input(f"[数字]{prompt}").strip())
        else:
            return float(input(f"[浮点数]{prompt}").strip())
    except ValueError:
        if convert_func == int:
            print("无效输入，请输入一个有效的整数")
        else:
            print("无效输入，请输入一个有效的浮点数")

model = joblib.load("student_failure_predict_model.joblib")
best_threshold = joblib.load("logreg_best_threshold.pkl")


student_id = input("学号 Student_ID: ")
age = get_valid_number("年龄 Age: ", int)
gender = get_valid_input("性别 Gender (Male/Female/Non-binary): ", ["Male", "Female", "Non-binary"])

education_level = get_valid_input("受教育程度 Education_Level (High School/College/University): ", ["High School", "College", "University"])
social_hours = get_valid_number("每日社交媒体时长(小时): ")
ai_hours = get_valid_number("每日AI工具使用时长(小时): ")
sleep_hours = get_valid_number("每日睡眠时长(小时): ")
sport_hours = get_valid_number("每日体育活动时长(小时): ")
mental_score = get_valid_number("心理健康评分(0-100): ")
physical_score = get_valid_number("身体健康评分(0-100): ")
isolation_score = get_valid_number("社会隔离评分(0-10): ")
burnout_level = get_valid_input("学业倦怠程度 Burnout_Level (Low/Moderate/High/Severe): ", ["Low", "Moderate", "High", "Severe"])
perf_score = get_valid_number("学业表现评分(0-100): ")

new_raw = pd.DataFrame({
    "Age": [age],
    "Gender": [gender],
    "Education_Level": [education_level],
    "Daily_Social_Media_Hours": [social_hours],
    "Daily_AI_Tool_Usage_Hours": [ai_hours],
    "Sleep_Hours": [sleep_hours],
    "Physical_Activity_Hours": [sport_hours],
    "Mental_Health_Score": [mental_score],
    "Physical_Health_Score": [physical_score],
    "Social_Isolation_Score": [isolation_score],
    "Burnout_Level": [burnout_level],
    "Academic_Performance_Score": [perf_score],
    "Student_ID": [student_id]
})


prob = model.predict_proba(new_raw)[:, 1][0]
pred = 1 if prob >= best_threshold else 0

print("\n===== 预测结果 =====")
print(f"学号：{student_id}")
print(f"预测标签(1=学业很有可能会失败,0=学业很有可能会成功）：{pred}")